In [41]:
import pandas as pd
import requests
import pandas as pd
import time
import json
import os
import sys
import pandas as pd
import re
from datetime import datetime
import pandas as pd
import numpy as np
import re


def map_action_numbers(old_df, new_df):
    """
    Map actionNumber from new_df to the appropriate rows in old_df based on time ranges and periods.
    
    Parameters:
    old_df (pandas.DataFrame): The old dataset with start_seconds and end_seconds columns
    new_df (pandas.DataFrame): The new dataset with actionNumber, period, and clock columns
    
    Returns:
    pandas.DataFrame: A copy of old_df with a new column 'actionNumber' added
    """
    # Make a copy of the old_df to avoid modifying the original
    result_df = old_df.copy()
    
    # Initialize the actionNumber column with NaN
    result_df['actionNumber'] = np.nan
    
    # Convert data types to ensure proper comparison
    result_df['PERIOD'] = result_df['PERIOD'].astype(int)
    result_df['start_seconds'] = result_df['start_seconds'].astype(float)
    result_df['end_seconds'] = result_df['end_seconds'].astype(float)
    
    new_df['period'] = new_df['period'].astype(int)
    new_df['actionNumber'] = new_df['actionNumber'].astype(int)
    
    # Helper function to convert clock format (PT12M00.00S) to seconds
    def clock_to_seconds(clock_str):
        if pd.isna(clock_str) or not isinstance(clock_str, str):
            return None
            
        # Extract minutes and seconds using regex
        minutes_match = re.search(r'PT(\d+)M', clock_str)
        seconds_match = re.search(r'M(\d+\.\d+)S', clock_str)
        
        if not seconds_match:
            seconds_match = re.search(r'M(\d+)S', clock_str)
            
        minutes = int(minutes_match.group(1)) if minutes_match else 0
        seconds = float(seconds_match.group(1)) if seconds_match else 0
        
        return minutes * 60 + seconds
    
    # Calculate game seconds for each action in the new dataset
    new_df['clock_seconds'] = new_df['clock'].apply(clock_to_seconds)
    new_df['period_start_seconds'] = (new_df['period'] - 1) * 720
    new_df['seconds_into_period'] = 720 - new_df['clock_seconds']
    new_df['game_seconds'] = new_df['period_start_seconds'] + new_df['seconds_into_period']
    
    # Sort new_df by period and game_seconds (should already be in order but just to be sure)
    new_df = new_df.sort_values(['period', 'game_seconds'])
    
    # Group by period for faster access
    new_df_by_period = {period: group for period, group in new_df.groupby('period')}
    
    # Function to find the nearest action number for a given time range and period
    def find_action_number(row):
        period = row['PERIOD']
        start_time = row['start_seconds']
        end_time = row['end_seconds']
        
        # Check if we have data for this period
        if period not in new_df_by_period:
            return np.nan
        
        period_data = new_df_by_period[period]
        
        # Find actions that fall within the time range
        matches = period_data[
            (period_data['game_seconds'] >= start_time) & 
            (period_data['game_seconds'] <= end_time)
        ]
        
        if not matches.empty:
            # Return the first action number in the time range
            # (actions are already ordered chronologically)
            return matches['actionNumber'].iloc[0]
        
        # If no direct match, find the closest action before the time range
        before_matches = period_data[period_data['game_seconds'] < start_time]
        if not before_matches.empty:
            return before_matches['actionNumber'].iloc[-1]
        
        # If still no match, find the closest action after the time range
        after_matches = period_data[period_data['game_seconds'] > end_time]
        if not after_matches.empty:
            return after_matches['actionNumber'].iloc[0]
        
        return np.nan
    
    # Apply the function to each row in the old dataset
    result_df['actionNumber'] = result_df.apply(find_action_number, axis=1)
    
    return result_df


def main():
    # Example usage
    old_df = pd.read_csv('old.csv')
    new_df = pd.read_csv('new.csv')
    
    result_df = map_action_numbers(old_df, new_df)
    
    # Save the result
    result_df.to_csv('mapped_results.csv', index=False)
    print(f"Mapping complete. Result saved to 'mapped_results.csv'")
    
    # Display some stats
    total_rows = len(result_df)
    mapped_rows = result_df['actionNumber'].notna().sum()
    mapping_percentage = (mapped_rows / total_rows) * 100
    
    print(f"Total rows in old dataset: {total_rows}")
    print(f"Successfully mapped rows: {mapped_rows} ({mapping_percentage:.2f}%)")


def pull_data(url):
    headers = {
        "Host": "stats.nba.com",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Referer": "https://stats.nba.com/",
        "Origin": "https://stats.nba.com",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
    }

    response = requests.get(url, headers=headers)
    json = response.json()
    
    # Handle the video events format
    if 'resultSets' in json and isinstance(json['resultSets'], dict):
        if 'Meta' in json['resultSets'] and 'videoUrls' in json['resultSets']['Meta']:
            video_urls = json['resultSets']['Meta']['videoUrls']
            playlist = json['resultSets'].get('playlist', [])
            
            # Convert video URLs to dataframe
            video_df = pd.DataFrame(video_urls)
            
            # Convert playlist to dataframe
            playlist_df = pd.DataFrame(playlist)
            
            # Merge video and playlist data
            if not playlist_df.empty:
                df = pd.concat([video_df, playlist_df], axis=1)
            else:
                df = video_df

        else:
            # Fallback in case no video data is found
            df = pd.DataFrame()

    # Handle the original stats format
    elif 'resultSets' in json and isinstance(json['resultSets'], list):
        if len(json["resultSets"]) == 1:
            data = json["resultSets"][0]["rowSet"]
            columns = json["resultSets"][0]["headers"]
            df = pd.DataFrame.from_records(data, columns=columns)
        else:
            data = json["resultSets"][1]["rowSet"]
            columns = json["resultSets"][1]["headers"]["columnNames"]
            df = pd.DataFrame.from_records(data, columns=columns)

    else:
        # Empty dataframe if no recognizable format is found
        df = pd.DataFrame()

    time.sleep(1.2)
    return df

import pandas as pd



# List of NBA team acronyms


# Example: Access the DataFrame for the Atlanta Hawks
# atl_df = team_dfs['ATL']

result_frames=[]
teams = [
    'ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 
    'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]

# Dictionary to store DataFrames for each team
team_dfs = {}

# Loop through each team and read the CSV file
for team in teams:
    file_path = f'2025/{team}_2025_clips_with_players.csv'
    
    if os.path.exists(file_path):  # Ensure the file exists before reading
        team_dfs[team] = pd.read_csv(file_path)
        print(f"Loaded {team}")
    else:
        print(f"File not found: {file_path}")


    all_df = pd.read_csv(file_path)

    # Identify GAME_IDs with at least one non-NaN URL
    valid_games = all_df[all_df['URL'].notna()]['GAMEID'].unique()

    # Filter out GAME_IDs without any non-NaN URLs
    missing_url_games = all_df[~all_df['GAMEID'].isin(valid_games)]['GAMEID'].unique()

    print("GAME_IDs without at least one non-NaN URL:")
    print(missing_url_games)


    missing=all_df[all_df.GAMEID.isin(missing_url_games)]
    missing

    # API endpoint

    all_rows = []
    for game_id in missing_url_games:
  


    # Direct API endpoint for play-by-play data

        url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_00{game_id}.json"

        # Set headers to mimic a browser request
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        }

        # Fetch the JSON data
        response = requests.get(url, headers=headers)


        if response.status_code == 200:
            data = response.json()

            actions = data.get('game', {}).get('actions', [])
                
                # Convert each action into a dictionary and add to the list
            for action in actions:
                action['game_id'] = game_id  # Add the game ID as a column
                all_rows.append(action)
            
        else:
            print(f"Failed to fetch data: {response.status_code}")
        time.sleep(1)
    time.sleep(1)
    teamdf = pd.DataFrame(all_rows)


    old_df=missing.copy()

    new_df=teamdf.copy()

    # Example usage
    old_df['GAMEID']='00'+old_df['GAMEID'].astype(str)
    old_df.sort_values(by='GAMEDATE',inplace=True)
    new_df.sort_values(by='timeActual',inplace=True)
    teamid=old_df['TEAM_ID'].iloc[0]
    new_df=new_df[new_df.teamId==teamid]
    result_df = map_action_numbers(old_df, new_df)
    
    # Save the result
    result_df.to_csv('mapped_results.csv', index=False)
    print(f"Mapping complete. Result saved to 'mapped_results.csv'")
    
    # Display some stats
    total_rows = len(result_df)
    mapped_rows = result_df['actionNumber'].notna().sum()
    mapping_percentage = (mapped_rows / total_rows) * 100
    
    print(f"Total rows in old dataset: {total_rows}")
    print(f"Successfully mapped rows: {mapped_rows} ({mapping_percentage:.2f}%)")

    result_frames.append(result_df)
data=pd.concat(result_frames)
data

Loaded ATL
GAME_IDs without at least one non-NaN URL:
[]


KeyError: 'timeActual'

In [3]:
data.sort_values(by=['GAMEDATE','GAMEID','PERIOD','start_seconds'],inplace=True)
data.to_csv('missing_actions.csv',index=False)
data['GAMEID'] = data['GAMEID'].str.replace(r'^00', '', regex=True)
data['GAMEID']= data['GAMEID'].astype(int)

In [39]:
import pandas as pd
import sys
data=pd.read_csv('missing_actions.csv')
action_map = pd.read_csv('nba_ping_results_final.csv')
action_map




action_map.rename(columns={'game_id':'GAMEID','action_number':'actionNumber'},inplace=True)
action_map=action_map[['GAMEID','actionNumber','url']]



newdata= data.merge(action_map,how='left',on=['GAMEID','actionNumber'])
newdata.drop(columns='URL',inplace=True)
newdata.rename(columns={'url':'URL'},inplace = True)

print(len(newdata[~newdata.URL.isna()]))



teams = [
    'ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 
    'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]

# Dictionary to store DataFrames for each team
team_dfs = {}

# Remove leading '00' from each game ID
#newdata['GAMEID'] = newdata['GAMEID'].str.lstrip('0').astype(int)
print(newdata['GAMEID'])
newdata['GAMEID']=newdata['GAMEID'].astype(int)
print(newdata['GAMEID'])
# Loop through each team and read the CSV file
for team in teams:
    file_path = f'2025/{team}_2025_clips_with_players.csv'
    df= pd.read_csv(file_path)
    print(len(df))
    teamid=df['TEAM_ID'].iloc[0]
   
    df['GAMEID']=df['GAMEID'].astype(int)
    print(df.GAMEID.unique())



 
    print(len(df.GAMEID.unique()))
    teamnew=newdata[newdata.TEAM_ID==teamid]
    
    # Remove duplicate columns by transposing, dropping duplicates, and transposing back
    teamnew = teamnew.loc[:, ~teamnew.columns.duplicated()].copy()
    teamnew.drop_duplicates(subset=['GAMEID','TEAM_ID','actionNumber'],inplace=True)

    # Verify the columns

    print(teamnew.columns)


    df=df[~df.GAMEID.isin(teamnew.GAMEID.unique())]
    teamnew['URL']=None
    print(len(df.GAMEID.unique()))
    print(len(teamnew.GAMEID.unique()))
    df = pd.concat([df,teamnew])
    df.sort_values(by=['GAMEDATE','PERIOD','start_seconds'],inplace=True)
    #df.drop(columns=['level_0','index'],inplace=True)
    df['URL'] = df['URL'].str.replace('320x180.mp4', '1280x720.mp4')

    df.to_csv(file_path,index=False)
    print(len(df.GAMEID.unique()))
 

20952
0        22400239
1        22400239
2        22400239
3        22400239
4        22400239
           ...   
31906    22401013
31907    22401013
31908    22401013
31909    22401013
31910    22401013
Name: GAMEID, Length: 31911, dtype: int64
0        22400239
1        22400239
2        22400239
3        22400239
4        22400239
           ...   
31906    22401013
31907    22401013
31908    22401013
31909    22401013
31910    22401013
Name: GAMEID, Length: 31911, dtype: int64
13280
[22400064 22400079 22400100 22400103 22400121 22400135 22400152 22400157
 22400171 22400185 22400198 22400001 22400012 22400239 22400250 22400258
 22400030 22400280 22400287 22400041 22400300 22400315 22400323 22400334
 22400350 22401202 22401229 22400370 22400378 22400395 22400413 22400427
 22400438 22400461 22400477 22400486 22400506 22400522 22400556 22400563
 22400587 22400602 22400612 22400623 22400639 22400656 22400675 22400686
 22400701 22400719 22400736 22400744 22400756 22400773 22400790 224008

/tmp/ipykernel_53932/3100453469.py:67: FutureWarning: The default value of regex will change from True to False in a future version.
  df['URL'] = df['URL'].str.replace('320x180.mp4', '1280x720.mp4')


68
14206
[22400061 22400073 22400089 22400104 22400119 22400132 22400141 22400157
 22400172 22400187 22400202 22400001 22400218 22400230 22400239 22400021
 22400028 22400271 22400281 22400047 22400307 22400316 22400319 22400335
 22400345 22401205 22401217 22400363 22400381 22400393 22400407 22400420
 22400437 22400449 22400464 22400471 22400487 22400507 22400528 22400542
 22400562 22400577 22400587 22400604 22400620 22400629 22400635 22400650
 22400666 22400683 22400698 22400710 22400719 22400728 22400748 22400759
 22400769 22400789 22400811 22400829 22400836 22400852 22400866 22400891
 22400900 22400918 22400929 22400945 22400946 22400958 22400960 22400968
 22400978 22400993 22400994]
75
Index(['index', 'ENDTIME', 'EVENTS', 'FG2A', 'FG2M', 'FG3A', 'FG3M',
       'GAMEDATE', 'GAMEID', 'NONSHOOTINGFOULSTHATRESULTEDINFTS',
       'OFFENSIVEREBOUNDS', 'OPPONENT', 'PERIOD', 'SHOOTINGFOULSDRAWN',
       'STARTSCOREDIFFERENTIAL', 'STARTTIME', 'STARTTYPE', 'TURNOVERS',
       'DESCRIPTION', '

/tmp/ipykernel_53932/3100453469.py:67: FutureWarning: The default value of regex will change from True to False in a future version.
  df['URL'] = df['URL'].str.replace('320x180.mp4', '1280x720.mp4')


75
13911
[22400064 22400077 22400098 22400113 22400124 22400136 22400151 22400158
 22400187 22400199 22400214 22400218 22400014 22400239 22400240 22400022
 22400027 22400276 22400284 22400297 22400043 22400305 22400317 22400320
 22400348 22401211 22401224 22400364 22400379 22400396 22400415 22400421
 22400436 22400459 22400465 22400478 22400495 22400513 22400531 22400546
 22400560 22400571 22400585 22400595 22400606 22400613 22400636 22400651
 22400663 22400689 22400711 22400720 22400719 22400737 22400758 22400772
 22400791 22400807 22400824 22400840 22400853 22400861 22400888 22400901
 22400912 22400930 22400940 22400945 22400956 22400960 22400968 22400978
 22400993 22400994 22401011]
75
Index(['index', 'ENDTIME', 'EVENTS', 'FG2A', 'FG2M', 'FG3A', 'FG3M',
       'GAMEDATE', 'GAMEID', 'NONSHOOTINGFOULSTHATRESULTEDINFTS',
       'OFFENSIVEREBOUNDS', 'OPPONENT', 'PERIOD', 'SHOOTINGFOULSDRAWN',
       'STARTSCOREDIFFERENTIAL', 'STARTTIME', 'STARTTYPE', 'TURNOVERS',
       'DESCRIPTION', '

...

/tmp/ipykernel_53932/3100453469.py:67: FutureWarning: The default value of regex will change from True to False in a future version.
  df['URL'] = df['URL'].str.replace('320x180.mp4', '1280x720.mp4')


75
14706
[22400070 22400084 22400110 22400115 22400130 22400148 22400160 22400182
 22400197 22400006 22400228 22400233 22400243 22400239 22400026 22400262
 22400264 22400038 22400298 22400304 22400311 22400055 22400339 22400357
 22401214 22401226 22400360 22400379 22400392 22400417 22400433 22400445
 22400458 22400481 22400490 22400506 22400523 22400533 22400546 22400569
 22400581 22400605 22400618 22400643 22400658 22400661 22400677 22400687
 22400707 22400725 22400719 22400740 22400751 22400768 22400781 22401004
 22400804 22400809 22400827 22400843 22400858 22400872 22400881 22400894
 22400905 22400924 22400929 22400950 22400945 22400966 22400960 22400978
 22400981 22400989 22400993 22400627]
76
Index(['index', 'ENDTIME', 'EVENTS', 'FG2A', 'FG2M', 'FG3A', 'FG3M',
       'GAMEDATE', 'GAMEID', 'NONSHOOTINGFOULSTHATRESULTEDINFTS',
       'OFFENSIVEREBOUNDS', 'OPPONENT', 'PERIOD', 'SHOOTINGFOULSDRAWN',
       'STARTSCOREDIFFERENTIAL', 'STARTTIME', 'STARTTYPE', 'TURNOVERS',
       'DESCRI

/tmp/ipykernel_53932/3100453469.py:67: FutureWarning: The default value of regex will change from True to False in a future version.
  df['URL'] = df['URL'].str.replace('320x180.mp4', '1280x720.mp4')


76
14175
[22400073 22400090 22400103 22400121 22400147 22400155 22400191 22400204
 22400213 22400224 22400012 22400237 22400239 22400246 22400028 22400272
 22400035 22400291 22400302 22400051 22400326 22400341 22400353 22401208
 22401217 22400362 22400382 22400400 22400412 22400429 22400443 22400456
 22400472 22400489 22400512 22400530 22400545 22400548 22400573 22400589
 22400598 22400610 22400983 22400645 22400657 22400665 22400674 22400690
 22400700 22400720 22400719 22400735 22400744 22400757 22400771 22400799
 22400815 22400824 22400839 22400860 22400877 22400894 22400915 22400932
 22400941 22400945 22400954 22400960 22400975 22400978 22400995 22400993
 22400627]
73
Index(['index', 'ENDTIME', 'EVENTS', 'FG2A', 'FG2M', 'FG3A', 'FG3M',
       'GAMEDATE', 'GAMEID', 'NONSHOOTINGFOULSTHATRESULTEDINFTS',
       'OFFENSIVEREBOUNDS', 'OPPONENT', 'PERIOD', 'SHOOTINGFOULSDRAWN',
       'STARTSCOREDIFFERENTIAL', 'STARTTIME', 'STARTTYPE', 'TURNOVERS',
       'DESCRIPTION', 'HTM', 'VTM', 'team

/tmp/ipykernel_53932/3100453469.py:67: FutureWarning: The default value of regex will change from True to False in a future version.
  df['URL'] = df['URL'].str.replace('320x180.mp4', '1280x720.mp4')


In [ ]:
team='ATL'
file_path = f'2025/{team}_2025_clips_with_players.csv'
df=pd.read_csv(file_path)
df.tail(50).URL
df.tail(50)

,ENDTIME,EVENTS,FG2A,FG2M,FG3A,FG3M,GAMEDATE,GAMEID,NONSHOOTINGFOULSTHATRESULTEDINFTS,OFFENSIVEREBOUNDS,...,TEAM_ID,Year,start_seconds,end_seconds,mid_seconds,players_on,opp_players_on,season,actionNumber,index
13230,01:21,MISS Okongwu 7' Driving Floating Jump Shot\nHO...,1,0,0,0,2025-03-18,22400993,0,0,...,1610612737,2025,1351.0,1359.0,1355.0,1627777|1629027|1630168|1630249|1630700,203552|1629610|1630182|1630585|1631109,2024-25,301.0,122353.0
13231,01:00,Daniels 1' Driving Layup (15 PTS)\n,1,1,0,0,2025-03-18,22400993,0,0,...,1610612737,2025,1362.0,1380.0,1371.0,1627777|1629027|1630168|1630249|1630700,1629610|1630182|1630585|1631111|1631217,2024-25,312.0,122352.0
13232,00:32,Diabaté BLOCK (2 BLK): MISS Young 26' 3PT Jump...,1,1,1,0,2025-03-18,22400993,0,1,...,1610612737,2025,1392.0,1408.0,1400.0,1627777|1629027|1630168|1630249|1630700,1629610|1630182|1630585|1631111|1631217,2024-25,302.0,122348.0
13233,00:00,Okongwu 2' Reverse Layup (15 PTS) (Daniels 3 A...,1,1,0,0,2025-03-18,22400993,0,0,...,1610612737,2025,1425.0,1440.0,1432.5,1629027|1629611|1630168|1630249|1630700,1629610|1630182|1630585|1631111|1631217,2024-25,367.0,122346.0
13234,11:18,Risacher 26' 3PT Jump Shot (6 PTS) (Young 4 AS...,0,0,1,1,2025-03-18,22400993,0,0,...,1610612737,2025,1464.0,1482.0,1473.0,1629027|1630168|1630700|1631243|1642258,203552|1629610|1630182|1631109|1641878,2024-25,340.0,122434.0
13235,10:35,Risacher 26' 3PT Jump Shot (9 PTS) (Daniels 4 ...,0,0,1,1,2025-03-18,22400993,0,0,...,1610612737,2025,1507.0,1525.0,1516.0,1629027|1630168|1630700|1631243|1642258,203552|1629610|1630182|1631109|1641878,2024-25,391.0,122433.0
13236,10:07,Risacher 26' 3PT Jump Shot (12 PTS) (Daniels 5...,0,0,1,1,2025-03-18,22400993,0,0,...,1610612737,2025,1542.0,1553.0,1547.5,1629027|1630168|1630700|1631243|1642258,203552|1629610|1630182|1631109|1641878,2024-25,353.0,122432.0
13237,09:39,Young Bad Pass Turnover (P4.T7)\n,0,0,0,0,2025-03-18,22400993,0,0,...,1610612737,2025,1565.0,1581.0,1573.0,1629027|1630168|1630700|1631243|1642258,203552|1629610|1630182|1631109|1641878,2024-25,364.0,122431.0
13238,09:06,MISS Okongwu 26' 3PT Jump Shot\nBaugh REBOUND ...,0,0,1,0,2025-03-18,22400993,0,0,...,1610612737,2025,1599.0,1614.0,1606.5,1629027|1630168|1630700|1631243|1642258,203552|1629610|1630182|1631109|1641878,2024-25,409.0,122429.0
13239,08:54,Daniels 3PT Jump Shot (18 PTS) (Risacher 2 AST)\n,0,0,1,1,2025-03-18,22400993,0,0,...,1610612737,2025,1619.0,1626.0,1622.5,1629027|1630168|1630700|1631243|1642258,203552|1629610|1630182|1631109|1641878,2024-25,371.0,122428.0
